## MLSys

这一道题旨在帮助你快速回顾一些在学习MLSys之前就应熟悉的概念和思想。你需要构建一个基础的 softmax 回归算法，以及一个简单的两层神经网络。你将分别使用原生 Python（借助 numpy 库）实现这些内容，并使用原生 C/C++ 实现 softmax 回归。过程中我们会提供一些关于各个函数实现方式的建议，但具体细节总体上由你自行决定。需要特别说明的是，在 Python 版本中应大量使用 numpy 的线性代数运算；显式使用循环通常会使代码比应有的速度慢很多。

**我们知道这道题目中有大量说明文字，尤其是开头部分，而需要编写的代码相对较少。即便如此，也 _请_ 认真阅读本文档的全部内容。**

下一个代码块将安装所需的库。

In [ ]:
!pip3 install pybind11
!pip3 install numdifftools

## 问题 1：基础 `add` 函数，以及测试基础

为了说明这些题目和自动评分系统的工作流程，我们先用实现一个简单的 `add` 函数作为示例。请注意，`/part0` 目录中结构如下：

    data/
        train-images-idx3-ubyte.gz
        train-labels-idx1-ubyte.gz
        t10k-images-idx3-ubyte.gz
        t10k-labels-idx1-ubyte.gz
    src/
        simple_ml.py
        simple_ml_ext.cpp
    tests/
        test_simple_ml.py
    Makefile

`data/` 目录包含本作业所需的数据（MNIST 数据集的副本）；`src/` 目录包含你要编写实现的源文件；`tests/` 目录包含用于在本地评估你的解答的测试；`Makefile` 则用于编译代码，与题目中的 C++ 部分有关。

本部分的第一个问题要求你实现 `simple_ml.add()` 函数。这个简单函数不会在其他地方使用，只是用于帮助你熟悉题目结构。查看 `src/simple_ml.py` 文件，你会看到下面的 `add()` 函数框架。

```python
def add(x, y):
    """ A trivial 'add' function you should implement to get used to the 
    autograder and submission system.  The solution to this problem is in
    the homework notebook.

    Args:
        x (Python number or numpy array)
        y (Python number or numpy array)

    Return:
        Sum of x + y
    """
    ### YOUR CODE HERE
    pass
    ### END YOUR CODE
```
每个文件中的文档字符串都定义了函数应实现的输入/输出映射。请养成仔细阅读它们的习惯，因为提交错误最常见的原因就是没有认真阅读规范。这个函数的实现方式应该非常明显：只需将 `pass` 语句替换为正确代码，即下面这样：

```python
def add(x, y):
    """ A trivial 'add' function you should implement to get used to the 
    autograder and submission system.  The solution to this problem is in the
    the homework notebook.

    Args:
        x (Python number or numpy array)
        y (Python number or numpy array)

    Return:
        Sum of x + y
    """
    ### YOUR CODE HERE
    return x + y
    ### END YOUR CODE
```
请在你的 `src/simple_ml.py` 文件中完成这一修改。

### 运行本地测试

现在你需要测试代码是否可以正常工作。本题目始终使用标准的代码单元测试工具，即 `pytest`。在 `src/simple_ml.py` 文件中写好正确代码后，运行下面的命令。

In [ ]:
!python3 -m pytest -k "add"

如果一切正常，你会看到一个测试成功通过。要了解这个测试的工作方式，请查看 `tests/test_simple_ml.py` 文件，尤其是 `test_add()` 函数：

```python
def test_add():
    assert add(5,6) == 11
    assert add(3.2,1.0) == 4.2
    assert type(add(4., 4)) == float
    np.testing.assert_allclose(add(np.array([1,2]), np.array([3,4])),
                               np.array([4,6]))
```

这段代码会针对你实现的函数运行一组单元测试。如果函数实现正确，上面的所有断言都应通过，也就是说代码执行时不会报错。反之，如果实现有误（例如把上面的 `x + y` 改成了 `x - y`），这些断言就会失败，`pytest` 会指出相应测试未通过。

In [ ]:
# in this example cell, we replaced "x + y" with "x - y" in simple_ml.add()
!python3 -m pytest -k "add"

如你所见，系统会给出错误并标明断言失败所在的行，你可以据此返回并调试自己的实现。**你应熟悉如何阅读和跟踪测试文件，以便更好地理解实现应有的行为。**

正确开发和使用单元测试对于现代软件开发至关重要。本题目的一个附带目标，是希望你熟悉软件开发中单元测试的典型用法。当然，这里并不完全要求你为了通过题目而自己编写测试，但你应该学会阅读我们提供的测试文件，从中理解函数应如何工作。我们也**强烈**建议你为自己的实现补充更多测试，尤其是当代码通过了本地测试、提交后却仍然失败时。

最后补充一点：如果你习惯用打印语句调试代码，请注意 **pytest 默认会捕获所有输出**。向 pytest 传入 `-s` 参数即可关闭该行为，让测试在所有情况下都显示全部输出。

## 问题 2：加载 MNIST 数据

现在你已经熟悉本地测试系统，请尝试实现 `src/simple_ml.py` 文件中的下一个函数：`parse_mnist_data()`。下面是文件中的函数声明。通常我们不会再次完整演示整个流程，但这里再演示一次。

```python
def parse_mnist(image_filename, label_filename):
    """ Read an images and labels file in MNIST format.  See this page:
    http://yann.lecun.com/exdb/mnist/ for a description of the file format.

    Args:
        image_filename (str): name of gzipped images file in MNIST format
        label_filename (str): name of gzipped labels file in MNIST format

    Returns:
        Tuple (X,y):
            X (numpy.ndarray[np.float32]): 2D numpy array containing the loaded 
                data.  The dimensionality of the data should be 
                (num_examples x input_dim) where 'input_dim' is the full 
                dimension of the data, e.g., since MNIST images are 28x28, it 
                will be 784.  Values should be of type np.float32, and the data 
                should be normalized to have a minimum value of 0.0 and a 
                maximum value of 1.0 (i.e., scale original values of 0 to 0.0 
                and 255 to 1.0).

            y (numpy.ndarray[dtype=np.uint8]): 1D numpy array containing the
                labels of the examples.  Values should be of type np.uint8 and
                for MNIST will contain the values 0-9.
    """
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE
```

现在你应该已经熟悉文档字符串的作用，并大致知道如何实现这个函数。首先访问 http://yann.lecun.com/exdb/mnist/ 或这个备用[链接](https://web.archive.org/web/20220509025752/http://yann.lecun.com/exdb/mnist/)（页面底部），阅读 MNIST 数据的二进制格式。然后编写一个加载器，读取这种类型的文件，并按照文档字符串中的规范返回 numpy 数组。如果实现遇到问题，请务必仔细阅读文档字符串。我们建议使用 Python 的 `struct` 模块，并结合 `gzip` 模块和 `numpy` 来实现该函数。

实现完成后，运行本地单元测试。

In [ ]:
!python3 -m pytest -k "parse_mnist"

## 问题 3：Softmax 损失

在 `src/simple_ml.py` 的 `softmax_loss()` 函数中，实现 softmax（即交叉熵）损失。

如注释所述，`softmax_loss()` 接收一个 logits 的**二维数组**（即一个批次中不同样本各自的 $k$ 维 logits），以及对应的一维真实标签数组，并应输出整个批次上的**平均** softmax 损失。最简洁的实现可以**不**使用任何循环，而应完全利用 numpy 的向量化操作完成计算。作为参考，最简洁的实现只有一行代码。

在“实际”实现 softmax 损失时，通常会缩放 logits 以防止数值溢出，但这里不必考虑这一点，即使不处理，作业其余部分也能正常工作。下面的代码会运行测试用例。

In [ ]:
!python3 -m pytest -k "softmax_loss"

## 问题 4：Softmax 回归的随机梯度下降
本题将为线性 softmax 回归实现随机梯度下降（SGD）。


利用这些梯度，实现 `softmax_regression_epoch()` 函数。该函数使用指定的学习率/步长 `lr` 和小批次大小 `batch`，执行一轮 SGD，也就是完整遍历一次数据集。如文档字符串所述，函数应原地修改 `Theta` 数组。实现后运行测试。

In [ ]:
!python3 -m pytest -k "softmax_regression_epoch and not cpp"

### 使用 softmax 回归训练 MNIST

虽然这不属于测试内容，但既然已经写好了代码，你也可以尝试用 SGD 训练一个完整的 MNIST 线性分类器。可以使用 `src/simple_ml.py` 文件中的 `train_softmax()` 函数；该函数已经由我们写好，你无需自行实现，不过可以查看它的具体工作方式。

你可以用下面的代码观察训练过程。作为参考，如下所示，我们的实现在 Colab 上运行约 3 秒，并达到 7.97% 的错误率。

In [ ]:
import sys
sys.path.append("src/")
from simple_ml import train_softmax, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")

train_softmax(X_tr, y_tr, X_te, y_te, epochs=10, lr=0.2, batch=100)

## 问题 5：两层神经网络的 SGD

现在你已经为线性分类器编写了 SGD，接下来考虑一个简单的两层神经网络。具体而言，对于输入 $x \in \mathbb{R}^n$，我们考虑如下不含偏置项的两层神经网络：
\begin{equation}
z = W_2^T \mathrm{ReLU}(W_1^T x)
\end{equation}
其中 $W_1 \in \mathbb{R}^{n \times d}$ 和 $W_2 \in \mathbb{R}^{d \times k}$ 表示网络权重，网络具有 $d$ 维隐藏单元；$z \in \mathbb{R}^k$ 表示网络输出的 logits。我们仍使用 softmax/交叉熵损失，因此要解决的优化问题是：
\begin{equation}
\min_{W_1, W_2} \;\; \frac{1}{m} \sum_{i=1}^m \ell_{\mathrm{softmax}}(W_2^T \mathrm{ReLU}(W_1^T x^{(i)}), y^{(i)}).
\end{equation}
或者复用符号，用矩阵 $X \in \mathbb{R}^{m \times n}$ 描述批量形式，也可写成：
\begin{equation}
\min_{W_1, W_2} \;\; \ell_{\mathrm{softmax}}(\mathrm{ReLU}(X W_1) W_2, y).
\end{equation}

利用链式法则，可以推导该网络的反向传播更新。

利用这些梯度，在 `src/simple_ml.py` 文件中编写 `nn_epoch()` 函数。与上一题相同，解答应原地修改 `W1` 和 `W2` 数组。实现后运行下面的测试。务必按照上面的表达式使用矩阵运算来实现函数；这会比循环**快得多**、效率更高，而且所需代码也少得多。

In [ ]:
!python3 -m pytest -k "nn_epoch"

### 训练完整的神经网络

和之前一样，虽然这不是通过自动评分的硬性要求，但看看你的神经网络函数能把 MNIST 分类器训练到什么水平会很有趣。与 softmax 回归类似，`simple_ml.py` 文件中提供了 `train_nn()` 函数，可使用多轮 SGD 训练这个两层网络。下面的示例代码会训练一个拥有 400 个隐藏单元的两层网络。

In [ ]:
import sys

# Reload the simple_ml module which has been cached from the earlier experiment
import importlib
import simple_ml
importlib.reload(simple_ml)

sys.path.append("src/")
from simple_ml import train_nn, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")
train_nn(X_tr, y_tr, X_te, y_te, hidden_dim=400, epochs=20, lr=0.2)

我们的实现在 Colab 上运行约需 30 秒；如上所示，它在 MNIST 上达到 1.89\% 的错误率。对于不到 20 行左右的代码来说，效果相当不错……

## 问题 6：用 C++ 实现 Softmax 回归

本part的最后一题要求你实现与问题 4 相同的函数，即执行一轮线性 softmax 回归；但这次要使用 C++，而不是 Python。严格来说，这里的实际实现更接近原始 C，不过我们会使用 C++ 特性，并借助 [pybind11](https://pybind11.readthedocs.io) 库构建 Python 接口。后续作业中你也会用它连接 C++ 与 Python。虽然还有其他方案，但 pybind11 是一个相当方便的接口库：它仅由头文件构成，并允许你在单个 C++ 源文件中实现完整的 Python/C++ 接口。

你将在 `src/simple_ml_ext.cpp` 文件中实现相关内容。先看一下文件的有关部分。你需要在下面这个函数中编写代码：

```cpp
void softmax_regression_epoch_cpp(const float *X, const unsigned char *y, 
								  float *theta, size_t m, size_t n, size_t k, 
								  float lr, size_t batch)
{
    /**
     * A C++ version of the softmax regression epoch code.  This should run a 
     * single epoch over the data defined by X and y (and sizes m,n,k), and
     * modify theta in place.  Your function will probably want to allocate
     * (and then delete) some helper arrays to store the logits and gradients.
     * 
     * Args:
     *     X (const float *): pointer to X data, of size m*n, stored in row 
     *          major (C) format
     *     y (const unsigned char *): pointer to y data, of size m
     *     theta (float *): pointer to theta data, of size n*k, stored in row
     *          major (C) format
     *     m (size_t): number of examples
     *     n (size_t): input dimension
     *     k (size_t): number of classes
     *     lr (float): learning rate / SGD step size
     *     batch (int): SGD minibatch size
     * 
     * Returns:
     *     (None)
     */

    /// YOUR CODE HERE
    
    /// END YOUR CODE
}
```

实现中的第二部分是实际提供 Python 接口的 pybind11 代码：
```cpp
PYBIND11_MODULE(simple_ml_ext, m) {
    m.def("softmax_regression_epoch_cpp", 
    	[](py::array_t<float, py::array::c_style> X, 
           py::array_t<unsigned char, py::array::c_style> y, 
           py::array_t<float, py::array::c_style> theta,
           float lr,
           int batch) {
        softmax_regression_epoch_cpp(
        	static_cast<const float*>(X.request().ptr),
            static_cast<const unsigned char*>(y.request().ptr),
            static_cast<float*>(theta.request().ptr),
            X.request().shape[0],
            X.request().shape[1],
            theta.request().shape[1],
            lr,
            batch
           );
    },
    py::arg("X"), py::arg("y"), py::arg("theta"), 
    py::arg("lr"), py::arg("batch"));
}
```
这段代码已经在文件中提供，请完全不要修改。对于感兴趣的同学，它本质上只是使用 pybind 的 numpy 接口，从给定输入中提取原始指针，然后调用对应的 `softmax_regression_epoch_cpp` 函数。

基于上述背景，实现 `softmax_regression_epoch_cpp`，完成与 Python 实现相同的更新。请注意，由于你只能访问原始数据，因此必须手动完成所有矩阵—向量乘法，而不能依赖 numpy 进行矩阵运算。**注意：本题目不要使用外部矩阵库，请自行编写乘法；它相对简单。** 完成后，可以使用下面的命令测试实现。

In [ ]:
!make
!python3 -m pytest -k "softmax_regression_epoch_cpp"

请注意，与之前的代码不同，这里必须先实际编译 C++ 扩展，才能运行并测试。只要代码中包含 C++ 组件，就需要执行这一步。

### 使用 C++ 版本训练完整的 softmax 回归分类器

最后，尝试使用“直接内存访问”的 C++ 版本训练完整的 softmax 回归分类器。如果之前的 Python 版本需要约 3 秒，那么这个版本应该快得惊人，对吧？

In [ ]:
import sys
sys.path.append("src/")

# Reload the simple_ml module to include the newly-compiled C++ extension
import importlib
import simple_ml
importlib.reload(simple_ml)

from simple_ml import train_softmax, parse_mnist

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz", 
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")

train_softmax(X_tr, y_tr, X_te, y_te, epochs=10, lr = 0.2, batch=100, cpp=True)

正如预期，结果与 Python 版本完全一致，但代码竟然……慢了大约 5 倍？！这是怎么回事？原来，你在 C++ 版本中编写的“手动”矩阵乘法代码效率极低。Python 本身虽然是一种较慢的解释型语言，但 numpy 底层使用由 C（或者信不信由你，更可能是 Fortran）编写的矩阵乘法，并经过高度优化，能够利用向量运算、不同处理器的缓存层级以及其他高效数值运算必不可少的特性。希望这个免试题能够充分激发你对MLSys的兴趣，或许在以后你甚至会编写一个能够相对高效执行这些运算的矩阵库，至少在某些特殊情况下如此——总体而言，想击败 numpy 确实并不容易。

接下来，我们将把同一个训练过程迁移到 GPU 上。


## 问题 7：用 CUDA 训练 MNIST（可选）

在上一题中，你使用 C++ 和原始指针实现了 softmax 回归，但朴素实现并不一定比 numpy 更快。本题将进一步把一轮 softmax 回归训练迁移到 NVIDIA GPU 上。你需要补全 `src/simple_ml_cuda.cu` 中两个 CUDA kernel 的关键部分，并通过 pybind11 从 Python 调用它们。

本题要求 Linux、NVIDIA GPU、CUDA Toolkit 和 `nvcc`。Apple Silicon 和没有 NVIDIA GPU 的电脑无法在本地运行 CUDA，可以使用提供 GPU 的 Linux 服务器或 Google Colab。没有可用 GPU 时，CUDA 测试会自动跳过。

我们仍然训练线性 softmax 回归模型。对于一个 minibatch，先计算

$$G = \operatorname{softmax}(X\theta)-\operatorname{one\_hot}(y),$$

再执行更新

$$\theta \leftarrow \theta - \frac{\eta}{b}X^T G.$$

文件已经提供显存申请、主机与设备之间的数据复制、kernel launch、错误检查和 pybind11 包装。你需要完成：

1. 在 `softmax_gradient_kernel` 中计算每个样本的 logits；
2. 使用数值稳定的 softmax，将结果原地转换为 `softmax - one_hot`；
3. 在 `update_theta_kernel` 中计算一个参数对应的 minibatch 梯度并更新参数。

所有数组均按 row-major 顺序存储。请先写出 `X[row, col]`、`theta[feature, class]` 和 `G[row, class]` 对应的一维下标，再开始编写 kernel。不要使用 cuBLAS、Thrust 或其他矩阵运算库；本题希望你理解最基础的 CUDA 线程索引与内存访问。


In [ ]:
!make cuda
!python3 -m pytest -k "softmax_regression_epoch_cuda"

### 使用 CUDA 版本训练完整的 MNIST softmax 回归分类器

通过测试后，运行下面的代码完成十轮训练。CUDA 函数和前面的 Python/C++ 函数一样，会原地修改 `theta`。由于这个基础实现每个 batch 都会启动多个 kernel，而且矩阵规模较小，它不一定能够击败高度优化的 numpy；这正是 MLSys 中需要分析的问题之一。


In [ ]:
import sys
import time
import numpy as np
sys.path.append("src/")

from simple_ml import parse_mnist, loss_err
from simple_ml_cuda import cuda_available, softmax_regression_epoch_cuda

assert cuda_available(), "No CUDA-capable GPU is available"

X_tr, y_tr = parse_mnist("data/train-images-idx3-ubyte.gz",
                         "data/train-labels-idx1-ubyte.gz")
X_te, y_te = parse_mnist("data/t10k-images-idx3-ubyte.gz",
                         "data/t10k-labels-idx1-ubyte.gz")
theta = np.zeros((X_tr.shape[1], y_tr.max() + 1), dtype=np.float32)

cuda_training_time = 0.0
for epoch in range(10):
    start = time.perf_counter()
    softmax_regression_epoch_cuda(X_tr, y_tr, theta, lr=0.2, batch=100)
    cuda_training_time += time.perf_counter() - start
    train_loss, train_error = loss_err(X_tr @ theta, y_tr)
    test_loss, test_error = loss_err(X_te @ theta, y_te)
    print(f"{epoch:02d} | train loss {train_loss:.5f} | "
          f"train error {train_error:.5f} | test error {test_error:.5f}")

print(f"CUDA training time: {cuda_training_time:.3f}s")

至此，你已经完成 Part1 的所有内容。恭喜你！
希望你已经对 MLSys 有了初步的了解，并对即将到来的面试做好了准备。